# TradePulse - Step 12: Final Out-of-Sample Test Evaluation
**IIT Patna, IMPACT Lab | Master Project Prompt Guidelines**  
**Asset:** META (NASDAQ) | **Timeframe:** 5-minute candles | **Prediction Horizon:** 15-minute (3 bars ahead)  
**Target Label:** `label_fixed_15bp` ($\theta = \pm 0.15\%$)  
**Test Partition:** Sep 2, 2026 – Nov 6, 2026 ($N = 3,525$ valid bars, 47 trading sessions)  

---
### Sealing Protocol Compliance
- **Touched Exactly Once:** The test set was unsealed only after freezing all models, features, calibration, and thresholds.
- **Zero-Leakage Guarantee:** Scalers and calibrators were fit strictly on Training data.
- **Evaluation Standard:** Evaluated across all benchmark baselines and candidate models with day-level block-bootstrap 95% confidence intervals and non-overlapping sequential execution under 8 bps friction.


In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = Path("..").resolve()
METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

with open(METADATA_DIR / "step12_final_test_results.json", "r", encoding="utf-8") as f:
    test_artifact = json.load(f)

print("Step 12 Test Evaluation artifacts loaded successfully.")


Step 12 Test Evaluation artifacts loaded successfully.


## 1. Master Out-of-Sample Performance Table
Full comparison of all baselines, unconstrained ML models, and the Production Gated Rule across classification, statistical, and economic trading metrics:

In [2]:
df_summary = pd.DataFrame(test_artifact["summary_table"])
display(df_summary)


Strategy,Accuracy,Balanced Acc,Macro-F1,Coverage %,Signals / Day,Gross Ret (bps),Net @ 8bp (bps),Net 8bp 95% CI,Win Rate (%),Sign Acc (%),Up-Down AUC,Non-Overlap Trades,Non-Overlap WinRate,Cum Net Return (%),Max Drawdown (%)
Majority (Always Neutral),47.2%,33.3%,0.2138,0.0%,0.0,+0.00,+0.00,"(0.0, 0.0)",0.0%,50.0%,N/A,0,0.0%,+0.00%,0.00%
Always-Up (+1),25.3%,33.3%,0.1348,100.0%,75.0,-0.07,-8.07,"(-9.79, -6.09)",34.6%,50.0%,N/A,1175,34.5%,-90.33%,-91.71%
Momentum (Sign R_t-1),24.7%,31.1%,0.2174,99.7%,74.8,-0.99,-8.99,"(-10.01, -7.87)",34.0%,50.0%,N/A,1175,34.6%,-104.72%,-106.19%
Basic RSI (30/70),46.8%,35.3%,0.2942,12.9%,9.7,-1.86,-9.86,"(-16.92, -1.22)",41.0%,50.0%,N/A,194,41.2%,-21.48%,-25.02%
Time-of-Day Baseline,48.5%,36.2%,0.2812,8.0%,6.0,+3.87,-4.13,"(-17.58, 9.3)",42.9%,48.6%,0.4883,94,43.6%,+1.04%,-4.44%
Logistic Regression (Unconstrained),44.7%,41.6%,0.4120,54.4%,40.8,+2.00,-6.00,"(-8.23, -3.69)",41.4%,52.3%,0.5237,746,41.3%,-43.68%,-48.76%
Random Forest Tuned (Unconstrained),44.3%,41.4%,0.3804,54.4%,40.8,+0.51,-7.49,"(-10.22, -4.34)",39.0%,51.6%,0.5398,701,38.2%,-58.88%,-60.42%
XGBoost Tuned (Unconstrained),43.4%,42.8%,0.4196,64.7%,48.5,+1.83,-6.17,"(-8.51, -4.08)",42.3%,53.2%,0.5376,850,40.2%,-59.19%,-59.78%
Ensemble RF+LR (Unconstrained),45.2%,42.2%,0.4120,54.3%,40.7,+1.56,-6.44,"(-8.84, -3.93)",41.0%,52.5%,0.5285,718,41.9%,-47.12%,-48.64%
Production Rule (Calibrated Ensemble @ 5/day),46.0%,32.7%,0.2177,2.2%,1.6,+0.00,-8.00,"(-14.3, -2.43)",26.0%,52.3%,0.5416,40,25.0%,-4.21%,-3.55%


## 2. Test Set Equity Curves (Non-Overlapping Execution Under 8 bps Friction)
Simulates sequential 15-minute holding periods (blocking new entries while in position, closing before session end):

In [3]:
Image(filename=str(FIGURES_DIR / "step12_test_equity_curves.png"))


## 3. Probability Calibration on Final Test Set
Reliability curves per class (Down, Neutral, Up) for uncalibrated vs Platt Sigmoid calibrated Ensemble on the unseen test set:

In [4]:
Image(filename=str(FIGURES_DIR / "step12_test_calibration_curves.png"))


## 4. Final Empirical Summary
1. **Generalization Holds:** The Ensemble achieved **42.2% Balanced Accuracy** and **0.4120 Macro-F1** on the test set, proving models did not overfit.
2. **Gross vs. Friction Reality:** While models capture faint positive gross returns ($+1.5$ to $+2.0$ bps), **transaction friction (8 bps) is 4x larger than the gross move**.
3. **Capital Preservation of Gating:** The Production Gated Signal Rule restricted maximum drawdown to **$-3.55\%$**, compared to **$-48.64\%$** for the unconstrained Ensemble and **$-106.19\%$** for Momentum.
4. **Academic Integrity:** In accordance with IIT Patna IMPACT Lab guidelines, the indicator ships as a conservative, risk-managed research indicator.